In [1]:
%cd /run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough

import os
import re
import pandas as pd
from itertools import cycle

LOG_ROOT = "logs"
RESULT_FILE = "result_summary.txt"

PHASE_PATTERN = re.compile(r"=+\s*(Train Phase|Test Phase)\s*=+")
LINE_PATTERN = re.compile(
    r"^(.*?)\s*-\s*"
    r"Acc\s*([\d.]+)\s*\|\s*"
    r"BalAcc\s*([\d.]+)\s*\|\s*"
    r"Sens\s*([\d.]+)\s*\|\s*"
    r"Spec\s*([\d.]+)\s*\|\s*"
    r"AUROC\s*([\d.]+)\s*\|\s*"
    r"pAUROC\s*([\d.]+)"
)

/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough


In [12]:
rows = []
for experiment in os.listdir(LOG_ROOT):
    experiment_metadata = experiment.split("_")
    if len(experiment_metadata) == 2:
        delta = "nodelta"
    else:
        delta = experiment_metadata[-1]
    exp_path = os.path.join(LOG_ROOT, experiment)
    result_path = os.path.join(exp_path, RESULT_FILE)

    if not os.path.isdir(exp_path) or not os.path.isfile(result_path):
        continue

    current_phase = None
    with open(result_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            phase_match = PHASE_PATTERN.search(line)
            if phase_match:
                current_phase = phase_match.group(1).replace(" Phase", "")
                continue

            metric_match = LINE_PATTERN.match(line)
            if metric_match and current_phase is not None:
                database = metric_match.group(1)

                rows.append({
                    "model": experiment_metadata[0],
                    "feature": experiment_metadata[1],
                    "delta": delta,
                    "phase": current_phase,
                    "database": database,
                    "accuracy": float(metric_match.group(2)),
                    "balanced_accuracy": float(metric_match.group(3)),
                    "sensitivity": float(metric_match.group(4)),
                    "specificity": float(metric_match.group(5)),
                    "auroc": float(metric_match.group(6)),
                    "pauroc": float(metric_match.group(7)),
                })

# Final dataframe
df = pd.DataFrame(rows)

# Optional: sorting for readability
df = df.sort_values(
    by=["model", "delta", "feature"]
).reset_index(drop=True)

df_train = df[df['phase'] == "Train"]
df_test = df[df['phase'] == "Test"]

In [3]:
df.to_csv("Last_Result.csv")

In [13]:
df_mean_db_train = (
    df_train
    .groupby(["model", "feature", "delta"], as_index=False)
    .mean(numeric_only=True)
)

df_mean_db_train.sort_values(by="auroc", ascending=False)

,model,feature,delta,accuracy,balanced_accuracy,sensitivity,specificity,auroc,pauroc
13,resnet34,gtgram,nodelta,0.822875,0.787825,0.787825,0.787825,0.877000,0.079825
16,resnet34,melspectogram,nodelta,0.806500,0.639900,0.639900,0.639900,0.853350,0.047850
22,resnet34,spectogram,nodelta,0.796575,0.626950,0.626950,0.626950,0.843350,0.015550
11,resnet34,gtgram,delta,0.789925,0.624700,0.624700,0.624700,0.832750,0.015475
14,resnet34,melspectogram,delta,0.793450,0.641425,0.641425,0.641425,0.817025,0.028900
12,resnet34,gtgram,deltadelta,0.789000,0.626225,0.626225,0.626225,0.806400,0.005650
20,resnet34,spectogram,delta,0.766325,0.636675,0.636675,0.636675,0.795875,0.063900
21,resnet34,spectogram,deltadelta,0.754225,0.612925,0.612925,0.612925,0.783425,0.081100
15,resnet34,melspectogram,deltadelta,0.780000,0.611450,0.611450,0.611450,0.763900,0.010650
19,resnet34,mfcc,nodelta,0.757425,0.585875,0.585875,0.585875,0.733400,0.009875


In [15]:
df_mean_db_train[df_mean_db_train['model'] == 'bilstm'].to_csv("temp_data.csv", index=False)

In [6]:
df_mean_db = (
    df_test
    .groupby(["model", "feature", "delta"], as_index=False)
    .mean(numeric_only=True)
)

df_sort = df_mean_db.sort_values(by="auroc", ascending=False)#.head(5)

In [7]:
df_sort

,model,feature,delta,accuracy,balanced_accuracy,sensitivity,specificity,auroc,pauroc
12,resnet34,gtgram,deltadelta,0.717000,0.649200,0.649200,0.649200,0.724033,0.000000
11,resnet34,gtgram,delta,0.707367,0.634633,0.634633,0.634633,0.703200,0.000000
13,resnet34,gtgram,nodelta,0.701600,0.631900,0.631900,0.631900,0.691967,0.000000
2,bilstm,gtgram,nodelta,0.669700,0.590600,0.590600,0.590600,0.681667,0.000000
19,resnet34,mfcc,nodelta,0.702367,0.609400,0.609400,0.609400,0.675633,0.012067
16,resnet34,melspectogram,nodelta,0.690300,0.606733,0.606733,0.606733,0.667100,0.000000
5,bilstm,melspectogram,nodelta,0.690533,0.601933,0.601933,0.601933,0.663967,0.000000
17,resnet34,mfcc,delta,0.706933,0.609200,0.609200,0.609200,0.663333,0.000967
15,resnet34,melspectogram,deltadelta,0.695633,0.612367,0.612367,0.612367,0.662933,0.000000
14,resnet34,melspectogram,delta,0.685767,0.606767,0.606767,0.606767,0.660000,0.000000


In [9]:
valid_voters = (
    df_sort.apply(
        lambda r: f"{r['model']}_{r['feature']}"
        if r['delta'] == "nodelta"
        else f"{r['model']}_{r['feature']}_{r['delta']}",
        axis=1,
    )
    .tolist()
)


In [10]:
valid_voters

['resnet34_gtgram_deltadelta',
 'resnet34_gtgram_delta',
 'resnet34_gtgram',
 'bilstm_gtgram',
 'resnet34_mfcc']

In [ ]:

    if not os.path.isdir(exp_path) or not os.path.isfile(result_path):
        continue

    current_phase = None

    with open(result_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            phase_match = PHASE_PATTERN.search(line)
            if phase_match:
                current_phase = phase_match.group(1).replace(" Phase", "")
                continue

            metric_match = LINE_PATTERN.match(line)
            if metric_match and current_phase is not None:
                database = metric_match.group(1)

                rows.append({
                    "experiment": experiment,
                    "phase": current_phase,
                    "database": database,
                    "accuracy": float(metric_match.group(2)),
                    "balanced_accuracy": float(metric_match.group(3)),
                    "sensitivity": float(metric_match.group(4)),
                    "specificity": float(metric_match.group(5)),
                    "auroc": float(metric_match.group(6)),
                    "pauroc": float(metric_match.group(7)),
                })




           experiment  phase              database  accuracy  \
0              bilstm   Test                 CIRDZ    0.3408   
1              bilstm   Test    TBCoda Logitudinal    0.6061   
2              bilstm   Test      TBCoda Solicited    0.6002   
3              bilstm   Test  TBScreen Logitudinal    0.7672   
4              bilstm   Test    TBScreen Solicited    0.7847   
..                ...    ...                   ...       ...   
330  tera_rankingnnpu   Test  TBScreen Logitudinal    0.4727   
331  tera_rankingnnpu  Train    TBCoda Logitudinal    0.4756   
332  tera_rankingnnpu  Train      TBCoda Solicited    0.4566   
333  tera_rankingnnpu  Train  TBScreen Logitudinal    0.4866   
334  tera_rankingnnpu  Train    TBScreen Solicited    0.0957   

     balanced_accuracy  sensitivity  specificity   auroc  pauroc  
0               0.5343       0.5343       0.5343  0.5550  0.0000  
1               0.6074       0.6074       0.6074  0.6550  0.0000  
2               0.6018       0

In [32]:
df_train = df[df['phase'] == "Train"]
df_test = df[df['phase'] == "Test"]

In [33]:
df_train.drop(columns=["phase"], inplace=True)
df_test.drop(columns=["phase"], inplace=True)

/tmp/ipykernel_3714943/2889416078.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train.drop(columns=["phase"], inplace=True)
/tmp/ipykernel_3714943/2889416078.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test.drop(columns=["phase"], inplace=True)


In [34]:
output_path = "all_result_train.xlsx"
with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    for db_name, df_db in df_train.groupby("database"):
        sheet_name = str(db_name)[:31]  # Excel sheet name limit
        df_db.to_excel(writer, sheet_name=sheet_name, index=False)


In [35]:
output_path = "all_result_test.xlsx"
with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    for db_name, df_db in df_test.groupby("database"):
        sheet_name = str(db_name)[:31]  # Excel sheet name limit
        df_db.to_excel(writer, sheet_name=sheet_name, index=False)


In [17]:
df_test['database'].unique()

array(['CIRDZ', 'TBCoda Logitudinal', 'TBCoda Solicited',
       'TBScreen Logitudinal', 'TBScreen Solicited'], dtype=object)

In [18]:
df_test['database'] == "CIRDZ"

0       True
1      False
2      False
3      False
4      False
       ...  
322    False
327     True
328    False
329    False
330    False
Name: database, Length: 179, dtype: bool

In [20]:
df_test.columns

Index(['experiment', 'phase', 'database', 'accuracy', 'balanced_accuracy',
       'sensitivity', 'specificity', 'auroc', 'pauroc'],
      dtype='object')

In [ ]:
df_test[]

,experiment,phase,database,accuracy,balanced_accuracy,sensitivity,specificity,auroc,pauroc
0,bilstm,Test,CIRDZ,0.3408,0.5343,0.5343,0.5343,0.5550,0.0000
1,bilstm,Test,TBCoda Logitudinal,0.6061,0.6074,0.6074,0.6074,0.6550,0.0000
2,bilstm,Test,TBCoda Solicited,0.6002,0.6018,0.6018,0.6018,0.6826,0.0000
3,bilstm,Test,TBScreen Logitudinal,0.7672,0.6192,0.6192,0.6192,0.7687,0.0303
4,bilstm,Test,TBScreen Solicited,0.7847,0.5189,0.5189,0.5189,0.7091,0.0000
...,...,...,...,...,...,...,...,...,...
322,tera_ranking_unfreeze_nnpu,Test,TBScreen Logitudinal,0.4688,0.5000,0.5000,0.5000,0.5858,0.0000
327,tera_rankingnnpu,Test,CIRDZ,0.8049,0.4836,0.4836,0.4836,0.3456,0.0000
328,tera_rankingnnpu,Test,TBCoda Logitudinal,0.4727,0.4859,0.4859,0.4859,0.3581,0.0000
329,tera_rankingnnpu,Test,TBCoda Solicited,0.4812,0.4912,0.4912,0.4912,0.3349,0.0000


In [9]:
df_train.to_csv("train_result.csv", index=False)
df_test.to_csv("test_result.csv", index=False)

In [3]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
from itertools import cycle

LOG_ROOT = "logs"
RESULT_FILE = "result_summary.txt"
OUT_CSV = "all_results.csv"
PLOT_DIR = "plots"

os.makedirs(PLOT_DIR, exist_ok=True)

# =======================
# Regex patterns
# =======================
PHASE_PATTERN = re.compile(r"=+\s*(Train Phase|Test Phase)\s*=+")
LINE_PATTERN = re.compile(
    r"^(.*?)\s*-\s*"
    r"Acc\s*([\d.]+)\s*\|\s*"
    r"BalAcc\s*([\d.]+)\s*\|\s*"
    r"Sens\s*([\d.]+)\s*\|\s*"
    r"Spec\s*([\d.]+)\s*\|\s*"
    r"AUROC\s*([\d.]+)\s*\|\s*"
    r"pAUROC\s*([\d.]+)"
)

rows = []

# =======================
# Parse logs
# =======================
for experiment in os.listdir(LOG_ROOT):
    exp_path = os.path.join(LOG_ROOT, experiment)
    result_path = os.path.join(exp_path, RESULT_FILE)

    if not os.path.isdir(exp_path) or not os.path.isfile(result_path):
        continue

    current_phase = None

    with open(result_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            phase_match = PHASE_PATTERN.search(line)
            if phase_match:
                current_phase = phase_match.group(1).replace(" Phase", "")
                continue

            metric_match = LINE_PATTERN.match(line)
            if metric_match and current_phase:
                rows.append({
                    "experiment": experiment,
                    "phase": current_phase,
                    "database": metric_match.group(1),
                    "accuracy": float(metric_match.group(2)),
                    "balanced_accuracy": float(metric_match.group(3)),
                    "sensitivity": float(metric_match.group(4)),
                    "specificity": float(metric_match.group(5)),
                    "auroc": float(metric_match.group(6)),
                    "pauroc": float(metric_match.group(7)),
                })

# =======================
# Save CSV
# =======================
df = pd.DataFrame(rows)
df = df.sort_values(
    by=["database", "phase", "experiment"]
).reset_index(drop=True)

df.to_csv(OUT_CSV, index=False)

print(f"[OK] Saved CSV → {OUT_CSV}")

# =======================
# Plotting
# =======================
METRICS = [
    "accuracy",
    "balanced_accuracy",
    "sensitivity",
    "specificity",
    "auroc",
    "pauroc",
]

LINESTYLES = cycle(["-", "--", "-.", ":"])
MARKERS = cycle(["o", "s", "D", "^", "v", "x", "*"])

# Ensure consistent ordering
databases = sorted(df["database"].unique())
experiments = sorted(df["experiment"].unique())
phases = sorted(df["phase"].unique())

for phase in phases:
    df_phase = df[df["phase"] == phase]

    for metric in METRICS:
        plt.figure(figsize=(12, 6))

        linestyle_cycle = cycle(["-", "--", "-.", ":"])
        marker_cycle = cycle(["o", "s", "D", "^", "v", "x", "*"])

        for exp in experiments:
            df_exp = df_phase[df_phase["experiment"] == exp]
            if df_exp.empty:
                continue

            df_exp = df_exp.set_index("database").reindex(databases)

            plt.plot(
                databases,
                df_exp[metric],
                label=exp,
                linestyle=next(linestyle_cycle),
                marker=next(marker_cycle),
            )

        plt.title(f"{metric.upper()} | {phase}")
        plt.xlabel("Database")
        plt.ylabel(metric)
        plt.xticks(rotation=30, ha="right")
        plt.grid(True, alpha=0.3)
        plt.legend(title="Experiment", bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout()

        fname = f"{metric}_{phase}.png"
        plt.savefig(os.path.join(PLOT_DIR, fname))
        plt.close()

print(f"[OK] Corrected plots saved → {PLOT_DIR}/")

[OK] Saved CSV → all_results.csv
[OK] Corrected plots saved → plots/
